In [1]:
# [Problem 1] Looking back on scratch



When implementing deep learning **from scratch** (without frameworks), here’s what you typically needed to do:

1. **Data preparation**

   * Load the dataset (e.g., Iris).
   * Normalize/standardize features.
   * Encode target labels (e.g., versicolor = 0, virginica = 1).

2. **Model architecture**

   * Decide number of layers and neurons.
   * Initialize weights and biases manually (often with small random values).

3. **Forward propagation**

   * Compute weighted sum (`z = Wx + b`).
   * Apply activation functions (e.g., sigmoid for binary classification).

4. **Loss calculation**

   * Define a loss function (e.g., cross-entropy).

5. **Backward propagation**

   * Compute gradients of loss with respect to weights and biases.
   * Implement chain rule manually.

6. **Parameter updates**

   * Update weights and biases using gradient descent (or another optimizer).
   * `W = W - learning_rate * dW`
   * `b = b - learning_rate * db`

7. **Training loop**

   * Repeat forward + backward + update steps for multiple **epochs**.
   * Shuffle/batch data if needed.

8. **Evaluation**

   * Calculate accuracy (compare predicted vs. true labels).
   * Possibly visualize loss curve.

---

## 🔹 Transition to TensorFlow (Low-Level API)

TensorFlow will:

* Handle **automatic differentiation** (backpropagation).
* Provide efficient **tensor operations**.
* Let you define and update variables easily.

But since you’re not using high-level APIs (`tf.keras`, `tf.estimator`), you’ll still **see all the moving parts**, just without coding derivatives by hand.

---

✅ So, in the Sprint you’ll **map each of the scratch steps** to **how TensorFlow implements them**.

For example:

* **Initialize weights** → `tf.Variable(tf.random.normal([...]))`
* **Epoch loop** → `for epoch in range(num_epochs):`
* **Forward pass** → matrix ops with `tf.matmul`, activations with `tf.sigmoid`
* **Loss** → `tf.nn.sigmoid_cross_entropy_with_logits`
* **Gradients** → `tape.gradient(loss, [W, b])`
* **Update** → `optimizer.apply_gradients(...)`

In [2]:
# [Problem 2] Consider the correspondence between scratch and TensorFlow

Here is the correspondence, summarized simply:

***

## Correspondence between Scratch Implementation and TensorFlow 1.x Code

| Scratch Component (What you need to do) | TensorFlow 1.x Implementation (How it's done) | Simple Summary |
| :--- | :--- | :--- |
| **I had to initialize the weights** | Done using **`tf.Variable(tf.random_normal(...))`** within the `weights` dictionary. | Weights and biases are created as **`tf.Variable`** objects to store the parameters that the model will learn. |
| **I needed an epoch loop** | Implemented using the standard **Python `for epoch in range(num_epochs)`** loop (line 120). | The outer training loop is still a standard Python `for` loop that iterates over the data multiple times. |
| **I needed a Forward Pass ($\mathbf{XW} + \mathbf{b}$ and Activation)** | Defined inside the **`example_net` function** using **`tf.matmul`** (matrix multiplication), **`tf.add`**, and **`tf.nn.relu`** (activation function). | The network structure (layers) is built by combining low-level TensorFlow math operations (`tf.matmul`, `tf.add`) and activation functions (`tf.nn.relu`). |
| **I needed a Loss Function** | Defined by **`loss_op = tf.reduce_mean(tf.nn.sigmoid_cross_entropy_with_logits(...))`** (line 103). | TensorFlow uses a single optimized operation (`sigmoid_cross_entropy_with_logits`) to calculate the objective function for binary classification. |
| **I needed Backward Propagation (Calculating Gradients)** | Handled implicitly by the **`optimizer.minimize(loss_op)`** call (line 106). | The **`AdamOptimizer`** (or any optimizer) automatically calculates the gradients and applies the updates when its `minimize` function is called. This is **Automatic Differentiation**. |
| **I needed to update parameters** | Performed by executing the **`train_op`** inside the session: **`sess.run(train_op, feed_dict=...)`** (line 126). | The optimizer operation (`train_op`) applies the calculated gradients to update the **`tf.Variable`** weights/biases. |
| **I needed to handle mini-batches** | Managed by the custom **`GetMiniBatch` class** and the inner **`for i, (mini_batch_x, mini_batch_y) in enumerate(get_mini_batch_train):`** loop (line 125). | Batching is managed manually using a custom Python iterator class, which is then fed into the TensorFlow graph for each training step. |
| **I needed to input data** | Data placeholders are defined by **`X = tf.placeholder("float", [None, n_input])`** and data is passed using the **`feed_dict`** argument during `sess.run` (line 126). | Data is fed into the static **computation graph** via **`tf.placeholder`** nodes and the **`feed_dict`** dictionary during execution. |
| **I needed a way to run the entire calculation** | The entire structure (the **Computation Graph**) is executed by creating a **`tf.Session`** and using the **`sess.run(...)`** command. | The TensorFlow **session** is the environment that executes the defined operations and manages the state of the variables. |

***



In [3]:
# [ Problem 3] Create a model of Iris using all three types of objective variables

In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
# Use TensorFlow 2.x or later
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers

# Load dataset
df = pd.read_csv("Iris.csv")

# --- MODIFICATION 1: REMOVE DATA FILTERING ---
# df = df[(df["Species"]=="Iris-versicolor") | (df["Species"]=="Iris-virginica")] # REMOVED

y = df["Species"]
X = df.loc[:, ["SepalLengthCm", "SepalWidthCm", "PetalLengthCm", "PetalWidthCm"]]
X = np.array(X)
y = np.array(y)

# --- MODIFICATION 2: CONVERT ALL 3 LABELS TO NUMBERS ---
y[y=="Iris-setosa"]=0      # NEW
y[y=="Iris-versicolor"]=1
y[y=="Iris-virginica"]=2
y=y.astype(np.int64)

# --- MODIFICATION 3: ONE-HOT ENCODING (OHE) for 3 classes ---
# Define n_classes early to use for OHE
n_classes = 3 # CHANGED from 1 to 3

# One-Hot Encode the labels (y is shape [N, 1] with values 0, 1, 2)
y_ohe = np.zeros((y.size, n_classes), dtype=np.float32)
y_ohe[np.arange(y.size), y.flatten()] = 1.0
y = y_ohe # y is now shape [N, 3]

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=0)

class GetMiniBatch:
    # ... (GetMiniBatch class remains unchanged, but now handles y of shape [N, 3])
    def __init__(self, X, y, batch_size=10, seed=0):
        self.batch_size=batch_size
        np.random.seed(seed)
        shuffle_index=np.random.permutation(np.arange(X.shape[0]))
        self.X=X[shuffle_index]
        self.y=y[shuffle_index]
        # Use integer division to avoid DeprecationWarning
        self._stop=np.ceil(X.shape[0]/self.batch_size).astype(np.int64)
    def __len__(self):
        return self._stop
    def __getitem__(self, item):
        p0=item*self.batch_size
        p1=item*self.batch_size+self.batch_size
        return self.X[p0:p1], self.y[p0:p1]
    def __iter__(self):
        self._counter=0
        return self
    def __next__(self):
        if self._counter>=self._stop:
            raise StopIteration()
        p0=self._counter*self.batch_size
        p1=self._counter*self.batch_size+self.batch_size
        self._counter+=1
        return self.X[p0:p1], self.y[p0:p1]


# Hyperparameter settings
learning_rate = 0.001
batch_size = 10
num_epochs = 100
n_hidden1 = 50
n_hidden2 = 100
n_input = X_train.shape[1]
n_samples = X_train.shape[0]
n_classes = 3

# train mini batch iterator
get_mini_batch_train = GetMiniBatch(X_train, y_train, batch_size=batch_size)

# --- MODIFICATION 6: Build model using Keras Sequential API ---
# Use tf.keras.Input to define the input shape
model = models.Sequential([
    tf.keras.Input(shape=(n_input,)), # Define input layer shape
    layers.Dense(n_hidden1, activation='relu'),
    layers.Dense(n_hidden2, activation='relu'),
    layers.Dense(n_classes, activation='softmax') # Use softmax for multi-class classification
])

# --- MODIFICATION 7: Define loss function and optimizer ---
# Use categorical_crossentropy for one-hot encoded labels
loss_object = tf.keras.losses.CategoricalCrossentropy()
optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)

# --- MODIFICATION 8: Define accuracy metric ---
# Use CategoricalAccuracy for one-hot encoded labels
train_accuracy_metric = tf.keras.metrics.CategoricalAccuracy()
val_accuracy_metric = tf.keras.metrics.CategoricalAccuracy()

# --- MODIFICATION 9: Training loop with tf.GradientTape ---
@tf.function
def train_step(x, y):
    with tf.GradientTape() as tape:
        predictions = model(x, training=True)
        loss = loss_object(y, predictions)
    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    train_accuracy_metric.update_state(y, predictions)
    return loss

@tf.function
def val_step(x, y):
    predictions = model(x, training=False)
    v_loss = loss_object(y, predictions)
    val_accuracy_metric.update_state(y, predictions)
    return v_loss


print("Starting training...")
for epoch in range(num_epochs):
    # Reset metrics at the start of each epoch
    train_accuracy_metric.reset_state()
    val_accuracy_metric.reset_state()

    total_loss = 0
    for i, (mini_batch_x, mini_batch_y) in enumerate(get_mini_batch_train):
        loss = train_step(mini_batch_x, mini_batch_y)
        total_loss += loss

    # Calculate average loss over the epoch
    avg_loss = total_loss / len(get_mini_batch_train)

    # Calculate validation loss and accuracy
    val_loss = val_step(X_val, y_val)


    print(f"Epoch {epoch+1}, loss: {avg_loss:.4f}, val_loss: {val_loss:.4f}, acc: {train_accuracy_metric.result():.3f}, val_acc: {val_accuracy_metric.result():.3f}")

print("\nTraining finished!")

# --- MODIFICATION 10: Evaluate on test set ---
test_loss_metric = tf.keras.metrics.CategoricalCrossentropy()
test_accuracy_metric = tf.keras.metrics.CategoricalAccuracy()

@tf.function
def test_step(x, y):
    predictions = model(x, training=False)
    test_loss_metric.update_state(y, predictions)
    test_accuracy_metric.update_state(y, predictions)

test_step(X_test, y_test)
print(f"test_loss: {test_loss_metric.result():.4f}, test_acc: {test_accuracy_metric.result():.3f}")

Starting training...
Epoch 1, loss: 1.2751, val_loss: 0.9474, acc: 0.385, val_acc: 0.708
Epoch 2, loss: 0.8977, val_loss: 0.8068, acc: 0.688, val_acc: 0.708
Epoch 3, loss: 0.7697, val_loss: 0.6933, acc: 0.688, val_acc: 0.708
Epoch 4, loss: 0.6618, val_loss: 0.6006, acc: 0.708, val_acc: 0.708
Epoch 5, loss: 0.5807, val_loss: 0.5367, acc: 0.729, val_acc: 0.792
Epoch 6, loss: 0.5215, val_loss: 0.4906, acc: 0.844, val_acc: 0.875
Epoch 7, loss: 0.4741, val_loss: 0.4570, acc: 0.885, val_acc: 0.917
Epoch 8, loss: 0.4363, val_loss: 0.4295, acc: 0.906, val_acc: 0.917
Epoch 9, loss: 0.4018, val_loss: 0.4078, acc: 0.927, val_acc: 0.917
Epoch 10, loss: 0.3726, val_loss: 0.3890, acc: 0.938, val_acc: 0.917
Epoch 11, loss: 0.3461, val_loss: 0.3722, acc: 0.948, val_acc: 0.917
Epoch 12, loss: 0.3202, val_loss: 0.3576, acc: 0.948, val_acc: 0.917
Epoch 13, loss: 0.2970, val_loss: 0.3435, acc: 0.958, val_acc: 0.917
Epoch 14, loss: 0.2753, val_loss: 0.3306, acc: 0.958, val_acc: 0.917
Epoch 15, loss: 0.2552

In [5]:
# [Problem 4] Create a model of House Prices

## Load the dataset

In [6]:
import pandas as pd

df_house = pd.read_csv("train.csv")

In [7]:
from sklearn.model_selection import train_test_split
import numpy as np

# Select features and target
X = df_house[['GrLivArea', 'YearBuilt']]
y = df_house['SalePrice']

# Convert to numpy arrays
X = np.array(X)
y = np.array(y)

# Split data into training (80%), validation (10%), and test (10%) sets
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=0)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=0)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_train shape: (1168, 2)
y_train shape: (1168,)
X_val shape: (146, 2)
y_val shape: (146,)
X_test shape: (146, 2)
y_test shape: (146,)


## Model architecture

In [8]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Determine the number of input features
n_input_house = X_train.shape[1]

# Create a Sequential model
model_house = Sequential([
    # Input layer
    Dense(64, activation='relu', input_shape=(n_input_house,)),
    # Hidden layer
    Dense(32, activation='relu'),
    # Output layer for regression
    Dense(1, activation='linear')
])

model_house.summary()

C:\Users\lelin\workspace\AIData_Engineer\Lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                      │ (None, 64)                  │             192 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,305 (9.00 KB)

 Trainable params: 2,305 (9.00 KB)

 Non-trainable params: 0 (0.00 B)

## Loss function and optimizer

In [9]:
from tensorflow.keras import optimizers, losses

# Choose appropriate loss function (MSE) and optimizer (Adam)
loss_object_house = losses.MeanSquaredError()
optimizer_house = optimizers.Adam(learning_rate=0.001)

# Compile the model
model_house.compile(optimizer=optimizer_house, loss=loss_object_house)

model_house.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                      │ (None, 64)                  │             192 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,305 (9.00 KB)

 Trainable params: 2,305 (9.00 KB)

 Non-trainable params: 0 (0.00 B)

## Training

In [10]:
from tensorflow.keras import metrics

# Define the number of epochs and batch size
num_epochs_house = 100
batch_size_house = 32 # Choose a common batch size

# Create a GetMiniBatch iterator for the training data
# We need to redefine GetMiniBatch or ensure it works with regression targets
# Since the previous GetMiniBatch was for classification, let's adapt it slightly
# or use a simpler approach for mini-batching in TensorFlow 2.x if preferred.
# However, to align with the original problem context of comparing to scratch,
# let's assume GetMiniBatch is available and adapted, or implement simple batching.

# Re-using the GetMiniBatch class from the previous problem, assuming it handles
# shuffling and batching correctly for regression data.
# We already have the GetMiniBatch class defined in a previous cell.

get_mini_batch_train_house = GetMiniBatch(X_train, y_train, batch_size=batch_size_house)

# Define MAE metric
train_mae_metric_house = metrics.MeanAbsoluteError()
val_mae_metric_house = metrics.MeanAbsoluteError()

# Training loop
print("Starting training for House Prices model...")
for epoch in range(num_epochs_house):
    # Reset metrics at the start of each epoch
    train_mae_metric_house.reset_state()
    val_mae_metric_house.reset_state()

    total_loss = 0
    num_batches = 0

    for mini_batch_x, mini_batch_y in get_mini_batch_train_house:
        num_batches += 1
        # Use tf.GradientTape for automatic differentiation
        with tf.GradientTape() as tape:
            # Forward pass
            predictions = model_house(mini_batch_x, training=True)
            # Calculate loss
            loss = loss_object_house(mini_batch_y, predictions)

        # Calculate gradients
        gradients = tape.gradient(loss, model_house.trainable_variables)

        # Apply gradients to update weights
        optimizer_house.apply_gradients(zip(gradients, model_house.trainable_variables))

        total_loss += loss.numpy() # Accumulate loss

        # Update training MAE metric
        train_mae_metric_house.update_state(mini_batch_y, predictions)


    # Calculate average training loss for the epoch
    avg_train_loss = total_loss / num_batches

    # Evaluate on validation set
    val_predictions = model_house(X_val, training=False)
    val_loss = loss_object_house(y_val, val_predictions)

    # Update validation MAE metric
    val_mae_metric_house.update_state(y_val, val_predictions)

    # Print epoch results
    print(f"Epoch {epoch+1}/{num_epochs_house}, Train Loss: {avg_train_loss:.4f}, Train MAE: {train_mae_metric_house.result():.2f}, Val Loss: {val_loss.numpy():.4f}, Val MAE: {val_mae_metric_house.result():.2f}")

print("\nTraining finished!")

Starting training for House Prices model...
Epoch 1/100, Train Loss: 38509129728.0000, Train MAE: 179459.72, Val Loss: 39824961536.0000, Val MAE: 183729.58
Epoch 2/100, Train Loss: 37501587456.0000, Train MAE: 176730.22, Val Loss: 38281506816.0000, Val MAE: 179606.83
Epoch 3/100, Train Loss: 35255943168.0000, Train MAE: 170506.14, Val Loss: 34974859264.0000, Val MAE: 170485.64
Epoch 4/100, Train Loss: 30928623616.0000, Train MAE: 157850.33, Val Loss: 29231613952.0000, Val MAE: 153435.77
Epoch 5/100, Train Loss: 24272498688.0000, Train MAE: 136191.16, Val Loss: 21265311744.0000, Val MAE: 126170.40
Epoch 6/100, Train Loss: 16188644352.0000, Train MAE: 104718.93, Val Loss: 12890438656.0000, Val MAE: 89829.12
Epoch 7/100, Train Loss: 9118546944.0000, Train MAE: 69481.17, Val Loss: 6958164992.0000, Val MAE: 58263.81
Epoch 8/100, Train Loss: 5224896000.0000, Train MAE: 47772.71, Val Loss: 4606840320.0000, Val MAE: 46467.44
Epoch 9/100, Train Loss: 4106007552.0000, Train MAE: 43949.06, Val Lo

## Evaluation

In [11]:
# Define appropriate regression metrics for the test set evaluation
test_mae_metric_house = tf.keras.metrics.MeanAbsoluteError()
test_rmse_metric_house = tf.keras.metrics.RootMeanSquaredError()

# Use the trained model_house to make predictions on the X_test data
test_predictions = model_house(X_test, training=False)

# Update the state of the defined test metrics using the true test labels (y_test) and the model's predictions
test_mae_metric_house.update_state(y_test, test_predictions)
test_rmse_metric_house.update_state(y_test, test_predictions)

# Calculate the test loss (MSE) for comparison, using the same loss object as training
test_loss = loss_object_house(y_test, test_predictions)

# Get the final result of the test metrics
test_mae_result = test_mae_metric_house.result().numpy()
test_rmse_result = test_rmse_metric_house.result().numpy()
test_loss_result = test_loss.numpy()

# Print the results of the test evaluation
print("\nTest Evaluation Results:")
print(f"Test Loss (MSE): {test_loss_result:.4f}")
print(f"Test MAE: {test_mae_result:.2f}")
print(f"Test RMSE: {test_rmse_result:.2f}")


Test Evaluation Results:
Test Loss (MSE): 3567655168.0000
Test MAE: 37001.75
Test RMSE: 59729.85


## Consider classification vs. regression


In [12]:
# [Problem 5] Comparison of classification and regression problems


In [13]:
import tensorflow as tf

print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
print("GPU Device Name:", tf.test.gpu_device_name())

Num GPUs Available:  0
GPU Device Name: 


## Load the mnist dataset



In [14]:
from tensorflow.keras.datasets import mnist

# Load the MNIST dataset
(x_train, y_train), (x_test, y_test) = mnist.load_data()

print("x_train shape:", x_train.shape)
print("y_train shape:", y_train.shape)
print("x_test shape:", x_test.shape)
print("y_test shape:", y_test.shape)

x_train shape: (60000, 28, 28)
y_train shape: (60000,)
x_test shape: (10000, 28, 28)
y_test shape: (10000,)


## Data preprocessing



In [15]:
import numpy as np
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

# 1. Normalize the image pixel values
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# 2. Reshape the images
x_train = x_train.reshape(x_train.shape[0], 784)
x_test = x_test.reshape(x_test.shape[0], 784)

# 3. One-hot encode the labels
num_classes = 10
y_train_ohe = to_categorical(y_train, num_classes)
y_test_ohe = to_categorical(y_test, num_classes)

# 4. Split the training data into training and validation sets
# Using 80% for training and 20% for validation
x_train_split, x_val_split, y_train_split, y_val_split = train_test_split(
    x_train, y_train_ohe, test_size=0.2, random_state=42
)

# 5. Print the shapes of the resulting sets
print("x_train_split shape:", x_train_split.shape)
print("y_train_split shape:", y_train_split.shape)
print("x_val_split shape:", x_val_split.shape)
print("y_val_split shape:", y_val_split.shape)
print("x_test shape:", x_test.shape)
print("y_test_ohe shape:", y_test_ohe.shape)

x_train_split shape: (48000, 784)
y_train_split shape: (48000, 10)
x_val_split shape: (12000, 784)
y_val_split shape: (12000, 10)
x_test shape: (10000, 784)
y_test_ohe shape: (10000, 10)


## Model architecture



In [16]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Instantiate a Sequential model
model_mnist = Sequential()

# Add the input layer
# Input shape is 784 (flattened 28x28 images)
model_mnist.add(Dense(128, activation='relu', input_shape=(784,)))

# Add hidden layers
model_mnist.add(Dense(64, activation='relu'))
model_mnist.add(Dense(32, activation='relu'))


# Add the output layer
# 10 neurons for 10 classes (digits 0-9) with softmax activation
model_mnist.add(Dense(10, activation='softmax'))

# Print the model summary
model_mnist.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                      │ (None, 128)                 │         100,480 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_7 (Dense)                      │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_8 (Dense)                      │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_9 (Dense)                      │ (None, 10)                  │             330 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 111,146 (434.16 KB)

 Trainable params: 111,146 (434.16 KB)

 Non-trainable params: 0 (0.00 B)

## Loss function and optimizer



In [17]:
from tensorflow.keras import optimizers, losses, metrics

# Choose appropriate loss function (Categorical Crossentropy) and optimizer (Adam)
loss_object_mnist = losses.CategoricalCrossentropy()
optimizer_mnist = optimizers.Adam(learning_rate=0.001)

# Compile the model
model_mnist.compile(optimizer=optimizer_mnist,
                    loss=loss_object_mnist,
                    metrics=['accuracy']) # Include accuracy as a metric

model_mnist.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                      │ (None, 128)                 │         100,480 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_7 (Dense)                      │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_8 (Dense)                      │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_9 (Dense)                      │ (None, 10)                  │             330 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 111,146 (434.16 KB)

 Trainable params: 111,146 (434.16 KB)

 Non-trainable params: 0 (0.00 B)

## Training




In [18]:
# Define the number of epochs and batch size
num_epochs = 10
batch_size = 128 # A common batch size for MNIST

print("Starting training for MNIST model...")
# Train the model using the fit method
history = model_mnist.fit(
    x_train_split, y_train_split,
    epochs=num_epochs,
    batch_size=batch_size,
    validation_data=(x_val_split, y_val_split),
    verbose=1 # Show training progress
)

print("\nTraining finished!")

Starting training for MNIST model...
Epoch 1/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 10s 15ms/step - accuracy: 0.8755 - loss: 0.4216 - val_accuracy: 0.9477 - val_loss: 0.1828
Epoch 2/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 10s 14ms/step - accuracy: 0.9558 - loss: 0.1526 - val_accuracy: 0.9598 - val_loss: 0.1359
Epoch 3/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 10s 14ms/step - accuracy: 0.9667 - loss: 0.1111 - val_accuracy: 0.9610 - val_loss: 0.1290
Epoch 4/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9752 - loss: 0.0846 - val_accuracy: 0.9703 - val_loss: 0.0967
Epoch 5/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - accuracy: 0.9794 - loss: 0.0685 - val_accuracy: 0.9689 - val_loss: 0.1001
Epoch 6/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 10s 15ms/step - accuracy: 0.9831 - loss: 0.0548 - val_accuracy: 0.9733 - val_loss: 0.0903
Epoch 7/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 11s 15ms/step - accuracy: 0.9859 - loss: 0.0458 - val_accuracy: 0.9719 - val_loss: 0.0978
Epoch 8/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 10s 15ms/step -

## Evaluation



In [19]:
# Evaluate the trained model on the test set
test_loss, test_accuracy = model_mnist.evaluate(x_test, y_test_ohe, verbose=0)

# Print the test evaluation results
print("\nTest Evaluation Results:")
print(f"Test Loss (Categorical Crossentropy): {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")


Test Evaluation Results:
Test Loss (Categorical Crossentropy): 0.0951
Test Accuracy: 0.9745


## Reproduce scratch model (optional)




In [20]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras import optimizers, losses, metrics

# Clear any previous model definition to redefine
model_mnist = None

# Instantiate a Sequential model
model_mnist = Sequential()

# Add the input layer (implicitly the first Dense layer with input_shape)
# Input shape is 784 (flattened 28x28 images)
# Let's use a single hidden layer with 128 neurons, similar to a common simple setup
model_mnist.add(Dense(128, activation='relu', input_shape=(784,)))

# Add a single hidden layer (simplifying from the previous Keras model)
# You could add more hidden layers here if a more complex scratch model was used.
# For this example, let's stick to one hidden layer to demonstrate simplification.
# model_mnist.add(Dense(64, activation='relu')) # Removed
# model_mnist.add(Dense(32, activation='relu')) # Removed


# Add the output layer
# 10 neurons for 10 classes (digits 0-9) with softmax activation
model_mnist.add(Dense(10, activation='softmax'))

# Choose appropriate loss function (Categorical Crossentropy) and optimizer (Adam)
loss_object_mnist = losses.CategoricalCrossentropy()
optimizer_mnist = optimizers.Adam(learning_rate=0.001)

# Compile the model
model_mnist.compile(optimizer=optimizer_mnist,
                    loss=loss_object_mnist,
                    metrics=['accuracy']) # Include accuracy as a metric

# Print the model summary
model_mnist.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_10 (Dense)                     │ (None, 128)                 │         100,480 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_11 (Dense)                     │ (None, 10)                  │           1,290 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 101,770 (397.54 KB)

 Trainable params: 101,770 (397.54 KB)

 Non-trainable params: 0 (0.00 B)